<a href="https://colab.research.google.com/github/Tanvir-tareq/Assembaly-code-8086/blob/main/WATER%20JUG%20PROBLEM%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Water Jug Problem Solver with Interactive Visualization in Google Colab
# This creates a complete web app that runs in your Colab notebook!

# Install required packages
!pip install -q ipywidgets matplotlib networkx
!pip install -q gradio
!jupyter nbextension enable --py widgetsnbextension

import gradio as gr
import matplotlib.pyplot as plt
import networkx as nx
from collections import deque
import time
import numpy as np
from matplotlib.patches import Rectangle, Circle, Polygon
import matplotlib.animation as animation
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

class WaterJugSolver:
    """Solves water jug problem using BFS algorithm"""

    def __init__(self):
        self.solution_path = []
        self.states_visited = 0
        self.solve_time = 0

    def bfs(self, x_capacity, y_capacity, target):
        """Breadth-First Search algorithm"""
        start_time = time.time()

        start = (0, 0)
        queue = deque()
        queue.append((start, [start]))
        visited = set()
        visited.add(start)

        while queue:
            (x, y), path = queue.popleft()
            self.states_visited += 1

            # Check if we found the solution
            if x == target or y == target:
                self.solution_path = path
                self.solve_time = time.time() - start_time
                return True

            # Generate all possible next states
            next_states = []

            # Rule 1: Fill jug 1
            if x < x_capacity:
                next_states.append((x_capacity, y))

            # Rule 2: Fill jug 2
            if y < y_capacity:
                next_states.append((x, y_capacity))

            # Rule 3: Empty jug 1
            if x > 0:
                next_states.append((0, y))

            # Rule 4: Empty jug 2
            if y > 0:
                next_states.append((x, 0))

            # Rule 5: Pour from jug 1 to jug 2
            if x > 0 and y < y_capacity:
                pour = min(x, y_capacity - y)
                next_states.append((x - pour, y + pour))

            # Rule 6: Pour from jug 2 to jug 1
            if y > 0 and x < x_capacity:
                pour = min(y, x_capacity - x)
                next_states.append((x + pour, y - pour))

            # Add unvisited states to queue
            for state in next_states:
                if state not in visited:
                    visited.add(state)
                    queue.append((state, path + [state]))

        self.solve_time = time.time() - start_time
        return False

    def get_action_description(self, prev_state, current_state, x_cap, y_cap):
        """Get description of action between states"""
        prev_x, prev_y = prev_state
        curr_x, curr_y = current_state

        if curr_x == x_cap and prev_x != x_cap and curr_y == prev_y:
            return f"Fill Jug 1 ({x_cap}L)"
        elif curr_y == y_cap and prev_y != y_cap and curr_x == prev_x:
            return f"Fill Jug 2 ({y_cap}L)"
        elif curr_x == 0 and prev_x != 0 and curr_y == prev_y:
            return "Empty Jug 1"
        elif curr_y == 0 and prev_y != 0 and curr_x == prev_x:
            return "Empty Jug 2"
        elif curr_x < prev_x and curr_y > prev_y:
            return f"Pour {prev_x - curr_x}L from Jug 1 to Jug 2"
        elif curr_x > prev_x and curr_y < prev_y:
            return f"Pour {prev_y - curr_y}L from Jug 2 to Jug 1"
        else:
            return "Initial State"

    def get_solution_details(self, x_capacity, y_capacity, target):
        """Get detailed solution with actions"""
        if not self.solution_path:
            return []

        details = []
        for i, (x, y) in enumerate(self.solution_path):
            if i == 0:
                action = "Start with empty jugs"
            else:
                action = self.get_action_description(
                    self.solution_path[i-1],
                    (x, y),
                    x_capacity,
                    y_capacity
                )

            target_reached = (x == target or y == target)
            details.append({
                'step': i,
                'jug1': x,
                'jug2': y,
                'action': action,
                'target_reached': target_reached
            })

        return details

def create_jug_visualization(x_capacity, y_capacity, x_amount, y_amount, step_info):
    """Create matplotlib visualization of jugs"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8))

    # Set up the figure
    fig.suptitle(f'Step {step_info["step"]}: {step_info["action"]}',
                 fontsize=16, fontweight='bold', y=1.02)

    # Colors
    water_color = '#4a9fff'
    jug_color = '#2e4a7a'
    bg_color = '#f8f9fa'

    # Set background
    fig.patch.set_facecolor(bg_color)
    ax1.set_facecolor(bg_color)
    ax2.set_facecolor(bg_color)

    # Jug 1 visualization
    ax1.set_xlim(-1, 1)
    ax1.set_ylim(0, max(x_capacity, y_capacity) + 2)

    # Draw jug 1
    jug_width = 0.6
    jug_height = x_capacity

    # Jug outline
    ax1.add_patch(Rectangle(
        (-jug_width/2, 0), jug_width, jug_height,
        fill=False, edgecolor=jug_color, linewidth=3
    ))

    # Water in jug 1
    water_height = (x_amount / x_capacity) * jug_height if x_capacity > 0 else 0
    ax1.add_patch(Rectangle(
        (-jug_width/2, 0), jug_width, water_height,
        color=water_color, alpha=0.8, edgecolor='darkblue', linewidth=2
    ))

    # Water level indicator
    ax1.plot([-jug_width/2, jug_width/2],
             [water_height, water_height],
             color='red', linewidth=2)

    # Labels for jug 1
    ax1.text(0, jug_height + 0.5,
             f'Jug 1: {x_capacity}L\nCurrent: {x_amount}L',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

    # Highlight if target is reached
    if step_info['target_reached']:
        ax1.text(0, -1, '🎯 TARGET REACHED!',
                 ha='center', fontsize=14, color='green', fontweight='bold')

    ax1.axis('off')
    ax1.set_title(f'Jug 1 ({x_capacity}L)', fontsize=14, pad=15)

    # Jug 2 visualization
    ax2.set_xlim(-1, 1)
    ax2.set_ylim(0, max(x_capacity, y_capacity) + 2)

    # Draw jug 2
    jug_width = 0.6
    jug_height = y_capacity

    # Jug outline
    ax2.add_patch(Rectangle(
        (-jug_width/2, 0), jug_width, jug_height,
        fill=False, edgecolor=jug_color, linewidth=3
    ))

    # Water in jug 2
    water_height = (y_amount / y_capacity) * jug_height if y_capacity > 0 else 0
    ax2.add_patch(Rectangle(
        (-jug_width/2, 0), jug_width, water_height,
        color=water_color, alpha=0.8, edgecolor='darkblue', linewidth=2
    ))

    # Water level indicator
    ax2.plot([-jug_width/2, jug_width/2],
             [water_height, water_height],
             color='red', linewidth=2)

    # Labels for jug 2
    ax2.text(0, jug_height + 0.5,
             f'Jug 2: {y_capacity}L\nCurrent: {y_amount}L',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

    ax2.axis('off')
    ax2.set_title(f'Jug 2 ({y_capacity}L)', fontsize=14, pad=15)

    plt.tight_layout()
    return fig

def create_state_space_graph(solution_path, x_capacity, y_capacity):
    """Create state space graph visualization"""
    fig, ax = plt.subplots(figsize=(10, 8))

    G = nx.DiGraph()

    # Add nodes and edges
    for i in range(len(solution_path) - 1):
        from_state = solution_path[i]
        to_state = solution_path[i + 1]

        G.add_node(f"{from_state[0]},{from_state[1]}")
        G.add_node(f"{to_state[0]},{to_state[1]}")

        # Determine edge label
        if to_state[0] == x_capacity and from_state[0] != x_capacity:
            label = f"Fill J1"
        elif to_state[1] == y_capacity and from_state[1] != y_capacity:
            label = f"Fill J2"
        elif to_state[0] == 0 and from_state[0] != 0:
            label = f"Empty J1"
        elif to_state[1] == 0 and from_state[1] != 0:
            label = f"Empty J2"
        elif to_state[0] < from_state[0]:
            label = f"Pour\n{from_state[0]-to_state[0]}L\nJ1→J2"
        else:
            label = f"Pour\n{from_state[1]-to_state[1]}L\nJ2→J1"

        G.add_edge(f"{from_state[0]},{from_state[1]}",
                   f"{to_state[0]},{to_state[1]}",
                   label=label)

    # Create layout
    pos = nx.spring_layout(G, seed=42)

    # Draw nodes
    node_colors = []
    for node in G.nodes():
        x, y = map(int, node.split(','))
        if x == 0 and y == 0:
            node_colors.append('lightgreen')  # Start
        elif x == solution_path[-1][0] and y == solution_path[-1][1]:
            node_colors.append('red')  # Goal
        else:
            node_colors.append('lightblue')

    nx.draw_networkx_nodes(G, pos, node_size=1500,
                          node_color=node_colors,
                          edgecolors='black', linewidths=2)

    # Draw edges
    nx.draw_networkx_edges(G, pos, edge_color='gray',
                          arrows=True, arrowsize=20,
                          connectionstyle="arc3,rad=0.1")

    # Draw labels
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

    # Draw edge labels
    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                                font_size=8, font_color='darkred')

    ax.set_title("State Space Graph (BFS Path)", fontsize=16, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()

    return fig

def solve_and_visualize(x_capacity, y_capacity, target):
    """Main function to solve and visualize water jug problem"""

    # Validate inputs
    if x_capacity <= 0 or y_capacity <= 0 or target <= 0:
        return "Error: All values must be positive integers!", None, None, None, None

    if target > max(x_capacity, y_capacity):
        return f"Error: Target ({target}L) cannot be larger than jug capacities!", None, None, None, None

    # Solve the problem
    solver = WaterJugSolver()
    solution_exists = solver.bfs(x_capacity, y_capacity, target)

    if not solution_exists:
        return "No solution exists for these parameters!", None, None, None, None

    # Get solution details
    solution_details = solver.get_solution_details(x_capacity, y_capacity, target)

    # Create output text
    output_text = f"""🎯 **SOLUTION FOUND!** 🎯

**Problem:**
- Jug 1 Capacity: {x_capacity}L
- Jug 2 Capacity: {y_capacity}L
- Target Volume: {target}L

**Statistics:**
- Total Steps: {len(solution_details) - 1}
- States Visited: {solver.states_visited}
- Solving Time: {solver.solve_time:.4f} seconds
- Optimal Path: {[f'({x},{y})' for (x, y) in solver.solution_path]}

**Solution Steps:**\n"""

    for detail in solution_details:
        step_text = f"Step {detail['step']}: "
        step_text += f"Jug1 = {detail['jug1']}L, Jug2 = {detail['jug2']}L"
        if detail['action']:
            step_text += f" → {detail['action']}"
        if detail['target_reached']:
            step_text += " 🎯"
        output_text += step_text + "\n"

    # Create visualizations for first, middle, and last steps
    if solution_details:
        # First step visualization
        fig1 = create_jug_visualization(
            x_capacity, y_capacity,
            solution_details[0]['jug1'],
            solution_details[0]['jug2'],
            solution_details[0]
        )

        # Middle step visualization
        mid_idx = len(solution_details) // 2
        fig2 = create_jug_visualization(
            x_capacity, y_capacity,
            solution_details[mid_idx]['jug1'],
            solution_details[mid_idx]['jug2'],
            solution_details[mid_idx]
        )

        # Last step visualization
        fig3 = create_jug_visualization(
            x_capacity, y_capacity,
            solution_details[-1]['jug1'],
            solution_details[-1]['jug2'],
            solution_details[-1]
        )

        # State space graph
        fig4 = create_state_space_graph(solver.solution_path, x_capacity, y_capacity)

        return output_text, fig1, fig2, fig3, fig4

    return output_text, None, None, None, None

# Create the Gradio interface
demo = gr.Interface(
    fn=solve_and_visualize,
    inputs=[
        gr.Number(label="Jug 1 Capacity (Liters)", value=4, minimum=1, maximum=50),
        gr.Number(label="Jug 2 Capacity (Liters)", value=3, minimum=1, maximum=50),
        gr.Number(label="Target Volume (Liters)", value=2, minimum=1, maximum=50)
    ],
    outputs=[
        gr.Textbox(label="Solution Details", lines=20),
        gr.Plot(label="Step 0 Visualization"),
        gr.Plot(label="Intermediate Step Visualization"),
        gr.Plot(label="Final Step Visualization"),
        gr.Plot(label="State Space Graph")
    ],
    title="🎯 Water Jug Problem Visualizer 🎯",
    description="""Solve the classic water jug puzzle using BFS algorithm with visualizations!
    Enter jug capacities and target volume, then see the solution with interactive visualizations.""",
    theme="soft",
    examples=[
        [4, 3, 2],  # Classic example
        [5, 3, 4],  # 5L & 3L -> 4L
        [7, 5, 6],  # 7L & 5L -> 6L
        [8, 6, 4],  # 8L & 6L -> 4L
    ]
)

# Launch the app
print("🚀 Launching Water Jug Problem Visualizer...")
print("📊 The interface will open below. Enter values and click Submit!")
print("💡 Try the examples or enter your own values!")

# Launch with sharing enabled (creates public link)
demo.launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.9 MB/s eta 0:00:00
Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK
🚀 Launching Water Jug Problem Visualizer...
📊 The interface will open below. Enter values and click Submit!
💡 Try the examples or enter your own values!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1110a64ee363ebd564.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# New Section